# Food.com Baseline Evaluation

Dieses Notebook führt unsere erste saubere Offline-Evaluation auf dem Food.com-Dataset aus. Wir behandeln das Problem als Top-N-Recommendation mit positivem implizitem Feedback.

Modellierungsannahme für diese erste Version:

- positives Feedback: `rating >= 4`
- nur User mit mindestens 2 positiven Interaktionen werden evaluiert
- Split: strict leave-last-out nach Zeitstempel

### Inhaltsübersicht:
1. Imports & Setup
2. Datapipeline & Preprocessing
3. Model Training
4. Offline Evaluation  
5. Retrieval Evaluation & ANN Analysis 
6. Learning-to-Rank (LTR)  
7. Slice & Hybrid Analysis  
8. Diagnostics & Explainability  
9. Experiment Logging



### To Do:
- Markdowns
- hübschere Tabellen
- ein paar Plots
- F8: deep recsys
- F9: sequential
- F10: graph
- F11 ...

## 1. Imports & Setup

Wir laden die Evaluationsfunktionen und die Modelle, setzen Seeds für Reproduzierbarkeit und definieren zentrale Laufzeit-Schalter. `RUN_CONTENT_BASED` ist standardmaessig aktiv, `RUN_KNN_ON_FULL_DATA` bleibt aus Performance-Gruenden deaktiviert.

In [1]:
import os
import sys
import csv
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "evaluation").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("=== PHASE 1: SETUP ===")
print("Current working directory:", cwd)
print("Project root used:", project_root)

from evaluation.split import leave_last_out_split
from evaluation.metrics import recall_at_k, ndcg_at_k, catalog_coverage_at_k, novelty_at_k
from evaluation.retrieval_metrics import summarize_retrieval_metrics, retrieval_latency_ms

from models.baselines import PopularityRecommender, RandomRecommender, TrendingRecommender
from models.content_based import ContentBasedRecommender, HybridMFContentRecommender
from models.knn import ItemItemKNN
from models.mf import BiasedMatrixFactorization, BPRMatrixFactorization
from models.ltr_ranker import PointwiseLTRRanker
from models.ann_retriever import ANNMatrixFactorizationRetriever
from models.sequential import MarkovRecommender
from models.graph_recommender import PersonalizedPageRankRecommender

np.random.seed(42)
random.seed(42)
K = 20
RUN_KNN_ON_FULL_DATA = False
RUN_CONTENT_BASED = True
CONTENT_CANDIDATE_ITEMS = 20000


=== PHASE 1: SETUP ===
Current working directory: /Users/isabellenachbaur/Documents/FHGR/6. semester/Recommender Systems/projekt/recommended-systems/notebooks
Project root used: /Users/isabellenachbaur/Documents/FHGR/6. semester/Recommender Systems/projekt/recommended-systems


## 2. Datapipeline & Preprocessing

Hier bauen wir den finalen Modellierungsdatensatz auf:

- Food.com-RAW-Dateien laden
- Ratings in positives implizites Feedback umwandeln
- nur evaluierbare User behalten
- Rezept-Metadaten joinen
- leave-last-out Split vorbereiten

In [2]:
print("=== PHASE 2: DATA PIPELINE ===")

interactions = pd.read_csv("../data/raw/RAW_interactions.csv")
recipes = pd.read_csv("../data/raw/RAW_recipes.csv")

interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")
recipes["submitted"] = pd.to_datetime(recipes["submitted"], errors="coerce")

# Positive implicit feedback: ratings 4 and 5.
positive_df = interactions[interactions["rating"] >= 4].copy()
positive_df = positive_df.rename(columns={"date": "date_time"})
positive_df = positive_df.sort_values(["user_id", "date_time"])

# Leave-last-out needs at least two positive interactions per user.
user_counts = positive_df["user_id"].value_counts()
valid_users = user_counts[user_counts >= 2].index
positive_df = positive_df[positive_df["user_id"].isin(valid_users)].copy()

recipes_model = recipes.rename(columns={"id": "recipe_id"}).copy()
df_model = positive_df.merge(recipes_model, on="recipe_id", how="inner")

print(f"Positive interactions after filtering: {len(df_model):,}")
print(f"Users for evaluation: {df_model['user_id'].nunique():,}")
print(f"Recipes in final dataset: {df_model['recipe_id'].nunique():,}")
print(f"Date range: {df_model['date_time'].min()} to {df_model['date_time'].max()}")

train_df, test_df = leave_last_out_split(df_model, user_col="user_id", time_col="date_time")

train_item_counts = train_df["recipe_id"].value_counts().to_dict()
total_train_interactions = len(train_df)
total_items = train_df["recipe_id"].nunique()
recipe_names = (
    recipes_model.drop_duplicates("recipe_id")
    .assign(name=lambda x: x["name"].fillna("Unknown Recipe"))
    .set_index("recipe_id")["name"]
    .to_dict()
)

print("\nPreview of final training data:")
display(train_df[["user_id", "recipe_id", "date_time", "rating", "name"]].head(5))

=== PHASE 2: DATA PIPELINE ===
Positive interactions after filtering: 875,778
Users for evaluation: 52,364
Recipes in final dataset: 206,817
Date range: 2000-01-25 00:00:00 to 2018-12-19 00:00:00
Sorting data chronologically to prevent time leakage...
Dropping users with tied final timestamps because a strict leave-last-out split is not identifiable: 8115 users
Splitting data: Leave-last-out...
Train Set: 707232 interactions
Test Set: 44249 interactions

Preview of final training data:


,user_id,recipe_id,date_time,rating,name
0,1533,17338,2002-02-19,4,zucchini and rice casserole
1,1533,24375,2002-04-23,5,costa rican stuffed tortilla
2,1533,10721,2002-05-02,5,orange yogurt cream
3,1533,23891,2002-05-06,5,parmesan fish in the oven
4,1533,24136,2002-05-20,5,fennel mashed potatoes


## 3. Model Training

Wir trainieren die Collaborative-Filtering-Baselines und zusätzlich einen Content-Based-Recommender. Für Content-Based begrenzen wir die Kandidatenmenge auf die populaersten Trainingsrezepte, damit TF-IDF und Cosine Scoring auf dem grossen Food.com-Katalog praktikabel bleiben.

`Sequential Baseline: First-Order Markov`

Zusätzlich zu den bisherigen Baselines wird ein First-Order-Markov-Modell evaluiert.
Dieses Modell nutzt die Reihenfolge der Interaktionen und empfiehlt häufige Nachfolger
des zuletzt gesehenen Rezepts. Dadurch kann geprüft werden, ob im Food.com-Datensatz
ein messbares Sequential Signal vorhanden ist.

`Graph-Based Baseline: Personalized PageRank`

Zusätzlich wird ein graphbasierter Recommender evaluiert.
Die Interaktionen werden als bipartiter User-Item-Graph interpretiert.
Über random-walk-ähnliche Multi-Hop-Pfade werden Kandidaten gefunden,
die strukturell nah am Zieluser liegen.

In [ ]:
print("\n=== PHASE 3: BASELINE, SEQUENTIAL, GRAPH & RETRIEVAL MODEL TRAINING ===")

pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col="recipe_id")

trend_model = TrendingRecommender(days_window=180)
trend_model.fit(train_df, item_col="recipe_id", time_col="date_time")

markov_model = MarkovRecommender()
markov_model.fit(
    train_df,
    user_col="user_id",
    item_col="recipe_id",
    time_col="date_time"
)

ppr_model = None
'''
ppr_model = PersonalizedPageRankRecommender(
    alpha=0.15,
    max_steps=3,
    n_walks=200,
    random_state=42
)
ppr_model.fit(train_df, user_col="user_id", item_col="recipe_id")
'''

rand_model = RandomRecommender()
rand_model.fit(train_df, item_col="recipe_id")

content_model = None
if RUN_CONTENT_BASED:
    content_candidate_ids = train_df["recipe_id"].value_counts().head(CONTENT_CANDIDATE_ITEMS).index
    content_items_df = recipes_model[recipes_model["recipe_id"].isin(content_candidate_ids)].copy()
    content_train_df = train_df[train_df["recipe_id"].isin(content_candidate_ids)].copy()

    content_model = ContentBasedRecommender(
        text_cols=["name", "tags", "ingredients", "description"],
        metadata_cols=["minutes", "n_steps", "n_ingredients"],
        min_df=2,
        ngram_range=(1, 1),
        time_col="date_time",
        recency_decay=0.0,
    )
    print(f"Training Content-Based model on top {len(content_candidate_ids):,} recipes by train popularity...")
    content_model.fit(
        content_train_df,
        content_items_df,
        user_col="user_id",
        item_col="recipe_id"
    )

knn_model = None
if RUN_KNN_ON_FULL_DATA:
    knn_model = ItemItemKNN(k_neighbors=100, shrinkage=10)
    knn_model.fit(train_df, item_col="recipe_id")
else:
    print("Skipping Item-Item kNN on the full Food.com dataset because the current implementation is too expensive at this item scale.")

mf_model = BiasedMatrixFactorization(k_factors=16, epochs=5)
mf_model.fit(train_df, item_col="recipe_id")

bpr_model = BPRMatrixFactorization(k_factors=32, epochs=5)
bpr_model.fit(train_df, item_col="recipe_id")

print("\nBuilding ANN retrieval index from learned BPR item embeddings...")

ann_bpr_model = ANNMatrixFactorizationRetriever(
    backend="auto",
    normalize=True
)

ann_bpr_model.fit_from_mf_model(bpr_model)

hybrid_model = None
if content_model is not None:
    hybrid_model = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=0.9,
        adaptive=True,
        sparse_threshold=5
    )

print("\nAll enabled models trained and ready for evaluation.")



=== PHASE 3: BASELINE, SEQUENTIAL, GRAPH & RETRIEVAL MODEL TRAINING ===
Training Popularity Baseline...
Learned popularity ranking for 185226 unique recipes.
Training Trending Baseline (last 180 days)...
Learned 795 trending recipes.
Training First-Order Markov Sequential Recommender...
Learned transitions for 180526 source recipes.
Training Graph-Based Personalized PageRank Recommender (alpha=0.15, max_steps=3, n_walks=200)...
Graph contains 44,249 users, 185,226 items, and 707,232 user-item edges.
Training Random Baseline...
Random candidate universe contains 185226 recipes.
Training Content-Based model on top 20,000 recipes by train popularity...
Training Content-Based Recommender...
Using text columns: ['name', 'tags', 'ingredients', 'description']
Using metadata columns: ['minutes', 'n_steps', 'n_ingredients']
Feature dimension: 15691
Vocabulary size: 15691
Missing rates by feature: {'name': 0.0, 'tags': 0.0, 'ingredients': 0.0, 'description': 0.02335, 'minutes': 0.0, 'n_steps': 

KeyboardInterrupt: 

## 4. Offline Evaluation

Jetzt evaluieren wir alle aktivierten Modelle auf demselben leave-last-out Split mit Recall@20, NDCG@20, Coverage@20 und Novelty@20. Content-Based wird dabei als eigene Baseline ausgewiesen.

In [ ]:
print("\n=== PHASE 4: RANKING EVALUATION ===")

results = {
    "Popularity": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Trending (180d)": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Markov Sequential": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    #"Graph PPR": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Random": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Content-Based": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Biased MF (SQ)": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "BPR MF": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "ANN BPR Retrieval": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Hybrid BPR+Content": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []}
}

models = {
    "Popularity": pop_model,
    "Trending (180d)": trend_model,
    "Markov Sequential": markov_model,
    #"Graph PPR": ppr_model,
    "Random": rand_model,
    "Content-Based": content_model,
    "Biased MF (SQ)": mf_model,
    "BPR MF": bpr_model,
    "ANN BPR Retrieval": ann_bpr_model,
    "Hybrid BPR+Content": hybrid_model
}

if ann_bpr_model is None:
    results.pop("ANN BPR Retrieval")
    models.pop("ANN BPR Retrieval")
    
if hybrid_model is None:
    results.pop("Hybrid BPR+Content")
    models.pop("Hybrid BPR+Content")

if content_model is None:
    results.pop("Content-Based")
    models.pop("Content-Based")

if knn_model is not None:
    results["Item-Item kNN"] = {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []}
    models["Item-Item kNN"] = knn_model

print("Preparing user histories...")
train_history = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

test_users = test_df["user_id"].unique()
print(f"Evaluating {len(test_users):,} users. This can take a while...")

for user in test_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        results[name]["all_recs"].append(recs)
        results[name]["recalls"].append(recall_at_k(recs, [true_item], k=K))
        results[name]["ndcgs"].append(ndcg_at_k(recs, [true_item], k=K))
        results[name]["novelties"].append(
            novelty_at_k(recs, train_item_counts, total_train_interactions, k=K)
        )

print("\n--- FINAL MACRO-AVERAGED RESULTS TABLE ---")
print(f"{'Model':<20} | {'Recall@20':<10} | {'NDCG@20':<10} | {'Coverage@20':<12} | {'Novelty@20'}")
print("-" * 75)

for name in models.keys():
    cov = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    mean_rec = np.mean(results[name]["recalls"])
    mean_ndcg = np.mean(results[name]["ndcgs"])
    mean_nov = np.mean(results[name]["novelties"])
    print(f"{name:<20} | {mean_rec:.4f}     | {mean_ndcg:.4f}     | {cov:.4%} | {mean_nov:.4f}")
    
if np.mean(results["Popularity"]["recalls"]) > 0.15:
    print("\nWARNING: Popularity baseline is very strong (>15% Recall). Check whether the split is too easy or too popular-item-heavy.")
else:
    print("\nSanity checks passed.")



=== PHASE 4: MASTER EVALUATION LOOP ===
Preparing user histories...
Evaluating 44,249 users. This can take a while...

--- FINAL MACRO-AVERAGED RESULTS TABLE ---
Model                | Recall@20  | NDCG@20    | Coverage@20  | Novelty@20
---------------------------------------------------------------------------
Popularity           | 0.0312     | 0.0125     | 0.0367% | 10.3203
Trending (180d)      | 0.0029     | 0.0018     | 0.0173% | 15.6854
Random               | 0.0001     | 0.0000     | 99.1464% | 18.3655
Content-Based        | 0.0083     | 0.0035     | 9.9376% | 14.4408
Biased MF (SQ)       | 0.0264     | 0.0090     | 0.1091% | 11.1796
BPR MF               | 0.0311     | 0.0117     | 0.0367% | 10.4011
Hybrid BPR+Content   | 0.0225     | 0.0083     | 9.4522% | 12.4656

Sanity checks passed.


In [ ]:
print("\n--- TWO-STAGE RECOMMENDER INTERPRETATION ---")

print("""
Stage 1: Candidate Generation (Retriever)
------------------------------------------
BPR MF:
- behavioral retrieval using collaborative filtering
- retrieves items from latent user-item interactions

ANN BPR Retrieval:
- approximate nearest-neighbor retrieval
- uses precomputed latent item embeddings
- simulates scalable production retrieval
      
Content-Based:
- semantic retrieval using recipe metadata
- retrieves items using ingredients, tags, and descriptions

Popularity:
- fallback candidate generator
- strong for sparse users and cold-start scenarios

Stage 2: Ranking
----------------
Hybrid BPR + Content:
- merges collaborative and content-based candidates
- reranks retrieved items using reciprocal-rank blending

Interpretation:
---------------
The ranker can only reorder what retrieval keeps alive.
If retrieval misses a relevant recipe, ranking cannot recover it.

This follows the standard two-stage recommender design:
cheap high-recall retrieval first, precise ranking second.
""")


--- TWO-STAGE RECOMMENDER INTERPRETATION ---

Stage 1: Candidate Generation (Retriever)
------------------------------------------
BPR MF:
- behavioral retrieval using collaborative filtering
- retrieves items from latent user-item interactions

Content-Based:
- semantic retrieval using recipe metadata
- retrieves items using ingredients, tags, and descriptions

Popularity:
- fallback candidate generator
- strong for sparse users and cold-start scenarios

Stage 2: Ranking
----------------
Hybrid BPR + Content:
- merges collaborative and content-based candidates
- reranks retrieved items using reciprocal-rank blending

Interpretation:
---------------
The ranker can only reorder what retrieval keeps alive.
If retrieval misses a relevant recipe, ranking cannot recover it.

This follows the standard two-stage recommender design:
cheap high-recall retrieval first, precise ranking second.



## 5. Retrieval Evaluation & ANN Analysis

In diesem Abschnitt analysieren wir die Candidate-Generation
als erste Stufe eines modernen Two-Stage-Recommender-Systems.

Dabei wird untersucht:
- ob relevante Items im Candidate-Set überleben,
- wie sich verschiedene Retrieval-Budgets auswirken,
- wie breit die Retrieval-Modelle den Item-Katalog abdecken,
- und wie sich ANN-basiertes Retrieval gegenüber vollständigem Scoring verhält.

Evaluiert werden:
- Candidate Recall@K
- Candidate Coverage
- durchschnittliche Candidate-Set-Größe
- Duplicate Rates
- Retrieval Latency

Zusätzlich wird ein ANN-Retriever (Approximate Nearest Neighbors)
auf den gelernten BPR-Embeddings evaluiert.
Dabei werden Item-Embeddings offline indexiert und
ähnliche Kandidaten effizient im Embedding-Space gesucht.

In [ ]:
# === RETRIEVAL EVALUATION & ANN ANALYSIS: Candidate Recall under different budgets ===
'''
This phase evaluates candidate generators independently from ranking quality.

ANN retrieval uses the learned BPR latent item vectors
and performs nearest-neighbor search in embedding space.

This follows the modern two-stage recommender architecture:
offline item indexing + online user retrieval.
'''

candidate_budgets = [50, 100, 300, 500]

retrieval_models = {
    "Popularity": pop_model,
    "Trending (180d)": trend_model,
    "Markov Sequential": markov_model,
    #"Graph PPR": ppr_model,
    "Random": rand_model,
    "Content-Based": content_model,
    "Biased MF (SQ)": mf_model,
    "BPR MF": bpr_model,
    "ANN BPR Retrieval": ann_bpr_model,
    "Hybrid BPR+Content": hybrid_model,
}

# Remove disabled models automatically
retrieval_models = {
    name: model
    for name, model in retrieval_models.items()
    if model is not None
}

true_items_by_user = test_df.set_index("user_id")["recipe_id"].to_dict()

retrieval_rows = []

for model_name, model in retrieval_models.items():
    print(f"Evaluating retrieval for {model_name}...")

    max_budget = max(candidate_budgets)
    all_candidate_lists = []

    for user in test_users:
        user_hist = train_history.get(user, set())

        candidates = model.recommend(
            user_id=user,
            user_history=user_hist,
            k=max_budget
        )

        all_candidate_lists.append(candidates)

    for budget in candidate_budgets:
        metrics = summarize_retrieval_metrics(
            all_candidate_lists=all_candidate_lists,
            true_items_by_user=true_items_by_user,
            user_ids=test_users,
            total_catalog_size=total_items,
            k=budget,
        )

        retrieval_rows.append({
            "model": model_name,
            "candidate_budget": budget,
            **metrics,
        })

retrieval_df = pd.DataFrame(retrieval_rows)

print("=== PHASE 5: RETRIEVAL EVALUATION RESULTS ===")
display(retrieval_df)

Evaluating retrieval for Popularity...
Evaluating retrieval for Trending (180d)...
Evaluating retrieval for Random...
Evaluating retrieval for Content-Based...
Evaluating retrieval for Biased MF (SQ)...
Evaluating retrieval for BPR MF...
Evaluating retrieval for Hybrid BPR+Content...
=== PHASE 5: RETRIEVAL EVALUATION RESULTS ===


,model,candidate_budget,candidate_recall_at_50,candidate_coverage_at_50,avg_candidate_set_size_at_50,mean_duplicate_rate_at_50,candidate_recall_at_100,candidate_coverage_at_100,avg_candidate_set_size_at_100,mean_duplicate_rate_at_100,candidate_recall_at_300,candidate_coverage_at_300,avg_candidate_set_size_at_300,mean_duplicate_rate_at_300,candidate_recall_at_500,candidate_coverage_at_500,avg_candidate_set_size_at_500,mean_duplicate_rate_at_500
0,Popularity,50,0.057493,0.000756,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Popularity,100,NaN,NaN,NaN,NaN,0.081516,0.001404,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Popularity,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.137043,0.003736,300.0,0.0,NaN,NaN,NaN,NaN
3,Popularity,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.171574,0.005669,500.0,0.0
4,Trending (180d),50,0.007209,0.000405,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Trending (180d),100,NaN,NaN,NaN,NaN,0.017673,0.000713,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Trending (180d),300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.023616,0.001965,300.0,0.0,NaN,NaN,NaN,NaN
7,Trending (180d),500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032475,0.003110,500.0,0.0
8,Random,50,0.000249,0.999989,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Random,100,NaN,NaN,NaN,NaN,0.000655,1.000000,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("\n=== PHASE 5.1: ANN SERVING LATENCY CHECK ===")
"This checks the serving-style latency of the ANN/vector-index retriever on a small user sample."

if ann_bpr_model is not None:
    latency_users = list(test_users[:1000])
    latencies = []

    for user in latency_users:
        user_hist = train_history.get(user, set())
        _, latency_ms = ann_bpr_model.recommend_with_latency(
            user_id=user,
            user_history=user_hist,
            k=300,
        )
        latencies.append(latency_ms)

    print(f"ANN backend: {ann_bpr_model.active_backend}")
    print(f"Mean latency@300: {np.mean(latencies):.3f} ms")
    print(f"Median latency@300: {np.median(latencies):.3f} ms")
    print(f"P95 latency@300: {np.percentile(latencies, 95):.3f} ms")
else:
    print("ANN retriever disabled.")


In [ ]:
print("\n=== PHASE 5.2: RETRIEVAL LATENCY CHECK ===")

sample_user = test_users[0]
sample_history = train_history.get(sample_user, set())

_, brute_force_latency = retrieval_latency_ms(
    bpr_model.recommend,
    sample_user,
    sample_history,
    k=300
)

_, ann_latency = retrieval_latency_ms(
    ann_bpr_model.recommend,
    sample_user,
    sample_history,
    k=300
)

print(f"BPR brute-force latency: {brute_force_latency:.2f} ms")
print(f"ANN retrieval latency: {ann_latency:.2f} ms")

## 6. Learning-to-Rank (LTR)

In diesem Abschnitt erweitern wir das bestehende Two-Stage-Recommender-System um einen einfachen Pointwise Learning-to-Rank Ranker.

Dafür werden:
- Candidate-Sets aus dem Hybrid Retriever erzeugt,
- Ranking-Features pro `(user, item)` erstellt,
- und ein logistischer Regressions-Ranker trainiert,
  der die Kandidaten neu ordnet.

Der Ranker kombiniert dabei:
- BPR-Scores,
- Content-Scores,
- Popularity-Features
- sowie User-Aktivität.

In [ ]:
print("\n=== PHASE 6.1: BUILDING LTR TRAINING TABLE ===")

ltr_rows = []

sample_users = test_users[:5000]

for user in sample_users:

    true_item = (
        test_df[test_df["user_id"] == user]["recipe_id"]
        .iloc[0]
    )

    user_hist = train_history.get(user, set())

    # Candidate retrieval
    candidates = hybrid_model.recommend(
        user,
        user_history=user_hist,
        k=100
    )

    for item in candidates:

        # Labels
        label = int(item == true_item)

        # --- Feature Engineering ---

        # Popularity feature
        popularity_score = pop_model.popularity_scores.get(item, 0)

        # BPR score
        bpr_score = bpr_model.score(user, item)

        # Content similarity
        content_score = content_model.score(user, item)

        # User activity
        user_activity = len(user_hist)

        ltr_rows.append({
            "user_id": user,
            "recipe_id": item,
            "label": label,

            # interaction features
            "bpr_score": bpr_score,
            "content_score": content_score,

            # item features
            "popularity_score": popularity_score,

            # user features
            "user_activity": user_activity,
        })

ltr_df = pd.DataFrame(ltr_rows)

print("LTR rows:", len(ltr_df))
display(ltr_df.head())


=== PHASE 6.1: BUILDING LTR TRAINING TABLE ===
LTR rows: 500000


,user_id,recipe_id,label,bpr_score,content_score,popularity_score,user_activity
0,1533,82102,0,3.382782,0.078134,557,119
1,1533,22782,0,3.373368,0.084425,640,119
2,1533,89204,0,3.372322,0.078266,829,119
3,1533,39087,0,3.360989,0.095993,786,119
4,1533,32204,0,3.357996,0.072442,745,119


In [ ]:
print("\n=== PHASE 6.2: TRAINING LTR RANKER ===")

feature_columns = [
    "bpr_score",
    "content_score",
    "popularity_score",
    "user_activity",
]

ltr_ranker = PointwiseLTRRanker()

ltr_ranker.fit(
    train_df=ltr_df,
    feature_columns=feature_columns,
    label_column="label"
)

print("LTR ranker trained.")


=== PHASE 6.2: TRAINING LTR RANKER ===
LTR ranker trained.


In [ ]:
print("\n=== PHASE 6.3: EVALUATING LTR RANKER ===")

ltr_recalls = []
ltr_ndcgs = []

for user in test_users[:5000]:

    true_item = (
        test_df[test_df["user_id"] == user]["recipe_id"]
        .iloc[0]
    )

    user_hist = train_history.get(user, set())

    candidates = hybrid_model.recommend(
        user,
        user_history=user_hist,
        k=100
    )

    feature_rows = []

    for item in candidates:

        feature_rows.append({
            "recipe_id": item,

            "bpr_score": bpr_model.score(user, item),
            "content_score": content_model.score(user, item),
            "popularity_score": pop_model.popularity_scores.get(item, 0),
            "user_activity": len(user_hist),
        })

    feature_df = pd.DataFrame(feature_rows)

    reranked = ltr_ranker.rerank(
        feature_df,
        top_k=20
    )

    ltr_recalls.append(
        recall_at_k(reranked, [true_item], k=20)
    )

    ltr_ndcgs.append(
        ndcg_at_k(reranked, [true_item], k=20)
    )

print("\nLTR RESULTS")
print("Recall@20:", np.mean(ltr_recalls))
print("NDCG@20:", np.mean(ltr_ndcgs))


=== PHASE 6.3: EVALUATING LTR RANKER ===

LTR RESULTS
Recall@20: 0.0222
NDCG@20: 0.009390378808648476


## 7. Slice & Hybrid Analysis

Zusätzlich zur globalen Offline-Evaluation analysieren wir die Modellperformance
auf unterschiedlichen Nutzer- und Item-Gruppen.

Dabei untersuchen wir:
- Unterschiede zwischen Sparse-, Medium- und Heavy-Usern,
- das Verhalten der Modelle bei Cold Items,
- sowie den Einfluss verschiedener Hybrid-Gewichtungen (`alpha`).

Ziel dieser Analysen ist es,
Popularity Bias, Cold-Start-Probleme
und Trade-offs zwischen Relevanz und Coverage
besser zu verstehen.

In [ ]:
print("\n--- 7.1 USER ACTIVITY SLICE ANALYSIS ---")

user_train_counts = train_df.groupby("user_id")["recipe_id"].count().to_dict()

def user_activity_group(user_id):
    n = user_train_counts.get(user_id, 0)
    if n <= 3:
        return "Sparse users (<=3)"
    elif n <= 10:
        return "Medium users (4-10)"
    else:
        return "Heavy users (>10)"

slice_rows = []

for user in test_users:
    group = user_activity_group(user)
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        slice_rows.append({
            "model": name,
            "slice": group,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

slice_df = pd.DataFrame(slice_rows)

slice_summary = (
    slice_df
    .groupby(["model", "slice"])
    .agg(
        users=("recall", "count"),
        recall_at_20=("recall", "mean"),
        ndcg_at_20=("ndcg", "mean")
    )
    .reset_index()
)

display(slice_summary)


--- 7.1 USER ACTIVITY SLICE ANALYSIS ---


,model,slice,users,recall_at_20,ndcg_at_20
0,BPR MF,Heavy users (>10),8569,0.029758,0.011201
1,BPR MF,Medium users (4-10),9716,0.032009,0.011005
2,BPR MF,Sparse users (<=3),25964,0.031120,0.012056
3,Biased MF (SQ),Heavy users (>10),8569,0.022990,0.007630
4,Biased MF (SQ),Medium users (4-10),9716,0.026554,0.008896
5,Biased MF (SQ),Sparse users (<=3),25964,0.027384,0.009556
6,Content-Based,Heavy users (>10),8569,0.002451,0.000914
7,Content-Based,Medium users (4-10),9716,0.004323,0.001837
8,Content-Based,Sparse users (<=3),25964,0.011747,0.005010
9,Hybrid BPR+Content,Heavy users (>10),8569,0.026841,0.010431


In [ ]:
print("\n--- 7.2 COLD ITEM SLICE ANALYSIS ---")

cold_items = ContentBasedRecommender.cold_item_set(
    train_df,
    test_df,
    item_col="recipe_id"
)

print(f"Cold items in test set: {len(cold_items)}")

cold_rows = []

for user in test_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]

    # only evaluate users whose test item is a cold item
    if true_item not in cold_items:
        continue

    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)

        cold_rows.append({
            "model": name,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

cold_df = pd.DataFrame(cold_rows)

if len(cold_df) == 0:
    print("No cold items found in test set under current split.")
else:
    cold_summary = (
        cold_df
        .groupby("model")
        .agg(
            users=("recall", "count"),
            recall_at_20=("recall", "mean"),
            ndcg_at_20=("ndcg", "mean")
        )
        .reset_index()
    )

    display(cold_summary)


--- 7.2 COLD ITEM SLICE ANALYSIS ---
Cold items in test set: 4633


,model,users,recall_at_20,ndcg_at_20
0,BPR MF,4869,0.0,0.0
1,Biased MF (SQ),4869,0.0,0.0
2,Content-Based,4869,0.0,0.0
3,Hybrid BPR+Content,4869,0.0,0.0
4,Popularity,4869,0.0,0.0
5,Random,4869,0.0,0.0
6,Trending (180d),4869,0.0,0.0


In [ ]:
print("\n--- 7.3 HYBRID FINAL ALPHA CHECK ---")

'''
sample_users = test_users[:1000]

Tested different alpha values for the Hybrid BPR+Content model 
- to see how the balance between collaborative and content-based retrieval affects performance. 

alpha_values = [0.3, 0.5, 0.7, 0.9]
alpha_rows = []

for alpha in alpha_values:
    temp_hybrid = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=alpha,
        adaptive=True,
        sparse_threshold=5
    )

    recalls = []
    ndcgs = []
    all_recs = []

    for user in sample_users:
        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
        user_hist = train_history.get(user, set())

        recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

        recalls.append(recall_at_k(recs, [true_item], k=K))
        ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
        all_recs.append(recs)

    alpha_rows.append({
        "alpha": alpha,
        "recall_at_20": np.mean(recalls),
        "ndcg_at_20": np.mean(ndcgs),
        "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
    })

alpha_df = pd.DataFrame(alpha_rows)
display(alpha_df)
'''


--- 7.3 HYBRID FINAL ALPHA CHECK ---


'\nsample_users = test_users[:1000]\n\nTested different alpha values for the Hybrid BPR+Content model \n- to see how the balance between collaborative and content-based retrieval affects performance. \n\nalpha_values = [0.3, 0.5, 0.7, 0.9]\nalpha_rows = []\n\nfor alpha in alpha_values:\n    temp_hybrid = HybridMFContentRecommender(\n        mf_model=bpr_model,\n        content_model=content_model,\n        alpha=alpha,\n        adaptive=True,\n        sparse_threshold=5\n    )\n\n    recalls = []\n    ndcgs = []\n    all_recs = []\n\n    for user in sample_users:\n        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]\n        user_hist = train_history.get(user, set())\n\n        recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)\n\n        recalls.append(recall_at_k(recs, [true_item], k=K))\n        ndcgs.append(ndcg_at_k(recs, [true_item], k=K))\n        all_recs.append(recs)\n\n    alpha_rows.append({\n        "alpha": alpha,\n        "recall_at_20

In [ ]:
print("\n--- 7.4 HYBRID FINAL ---")

sample_users = test_users[:1000]

# tested: [0.3, 0.5, 0.7, 0.9]--> best result: alpha = 0.9
# reason: best Recall@20 + NDCG@20 with strong coverage
# interpretation: Food.com benefits more from stronger collaborative filtering (BPR) than from stronger content weighting.

# Therefore, the final hybrid model uses:

alpha = 0.9

temp_hybrid = HybridMFContentRecommender(
    mf_model=bpr_model,
    content_model=content_model,
    alpha=alpha,
    adaptive=True,
    sparse_threshold=5
)

recalls = []
ndcgs = []
all_recs = []

for user in sample_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

    recalls.append(recall_at_k(recs, [true_item], k=K))
    ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
    all_recs.append(recs)

alpha_df = pd.DataFrame([{
    "alpha": alpha,
    "recall_at_20": np.mean(recalls),
    "ndcg_at_20": np.mean(ndcgs),
    "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
}])

display(alpha_df)


--- 7.4 HYBRID FINAL ---


,alpha,recall_at_20,ndcg_at_20,coverage_at_20
0,0.9,0.016,0.006276,0.020688


## 8. Diagnostics And Explainability

Neben den Metriken schauen wir auf kurze Diagnostics: Content-Based-Erklaerungen, optional kNN-Erklaerungen, BPR-Bias und BPR-Latent-Space-Nachbarn.

In [ ]:
print("\n=== PHASE 8: DIAGNOSTICS & EXPLAINABILITY ===")

if content_model is not None:
    print("\n--- 8A. CONTENT-BASED EXPLAINABILITY ---")
    cb_users = [u for u in train_df["user_id"].value_counts().index if u in content_model.user_profiles]
    sample_user = cb_users[0]
    explained_cb_recs = content_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    print(f"Generating content-based explanations for User {sample_user}:")
    for i, (rec_id, explanation) in enumerate(explained_cb_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 8A. CONTENT-BASED EXPLAINABILITY ---")
    print("Skipped because Content-Based was disabled.")

if knn_model is not None:
    print("\n--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    heavy_users = train_df["user_id"].value_counts()
    sample_user = heavy_users[heavy_users > 5].sample(1, random_state=42).index[0]
    print(f"Generating explanations for User {sample_user}:")
    explained_recs = knn_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    for i, (rec_id, explanation) in enumerate(explained_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    print("Skipped because Item-Item kNN was disabled for the full Food.com run.")

print("\n--- 8C. BPR BIAS INSPECTION ---")
top_bias_indices = np.argsort(bpr_model.b_i)[-5:][::-1]
print("Top 5 Recipes by Learned Bias:")
for idx in top_bias_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Bias: {bpr_model.b_i[idx]:.4f})")

print("\n--- 8D. BPR LATENT SPACE INSPECTION ---")
anchor_id = train_df["recipe_id"].value_counts().index[0]
anchor_idx = bpr_model.item_mapping[anchor_id]
anchor_vector = bpr_model.Q[anchor_idx].reshape(1, -1)

similarities = cosine_similarity(anchor_vector, bpr_model.Q)[0]
nearest_indices = [idx for idx in np.argsort(similarities)[::-1] if idx != anchor_idx][:5]

print(f"Anchor recipe: {recipe_names.get(anchor_id, 'Unknown Recipe')}")
print("Nearest latent neighbors:")
for idx in nearest_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Latent Sim: {similarities[idx]:.4f})")



=== PHASE 8: DIAGNOSTICS & EXPLAINABILITY ===

--- 8A. CONTENT-BASED EXPLAINABILITY ---
Generating content-based explanations for User 37449:
1. garlic potatoes
   -> Recommended because it overlaps on: tags: low, tags: or, tags: fat.
2. roasted green beans with garlic and onions
   -> Recommended because it overlaps on: tags: low, tags: fat, tags: healthy.
3. simple cucumbers
   -> Recommended because it overlaps on: tags: low, tags: or, tags: fat.

--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---
Skipped because Item-Item kNN was disabled for the full Food.com run.

--- 8C. BPR BIAS INSPECTION ---
Top 5 Recipes by Learned Bias:
   -> kittencal s moist cheddar garlic oven fried chicken breast (Bias: 3.3831)
   -> crock pot chicken with black beans   cream cheese (Bias: 3.3711)
   -> jo mama s world famous spaghetti (Bias: 3.3710)
   -> creamy cajun chicken pasta (Bias: 3.3656)
   -> whatever floats your boat  brownies (Bias: 3.3561)

--- 8D. BPR LATENT SPACE INSPECTION ---
Anchor recipe: 

## 9. Experiment Logging

Zum Schluss schreiben wir alle aktivierten Modelllaeufe in `runs/runs.csv`, damit wir spaeter verschiedene Varianten vergleichen koennen.

In [ ]:
print("\n=== PHASE 9: EXPERIMENT LOGGING ===")

runs_file = "../runs/runs.csv"
os.makedirs("../runs", exist_ok=True)

run_rows = []
for name, model in models.items():
    coverage = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    hyperparams = ""
    if name == "BPR MF":
        hyperparams = f"k_factors={bpr_model.k}, epochs={bpr_model.epochs}, reg={bpr_model.reg}"
    elif name == "Biased MF (SQ)":
        hyperparams = f"k_factors={mf_model.k}, epochs={mf_model.epochs}, reg={mf_model.reg}"
    elif name == "ANN BPR Retrieval":
        hyperparams = f"source=BPR MF, backend={ann_bpr_model.active_backend}, include_bias={ann_bpr_model.include_item_bias}, overfetch={ann_bpr_model.overfetch_factor}"
    elif name == "Content-Based":
        hyperparams = f"top_items={CONTENT_CANDIDATE_ITEMS}, text=name/tags/ingredients/description, min_df={content_model.min_df}"
    elif name == "Trending (180d)":
        hyperparams = f"days_window={trend_model.days_window}"

    run_rows.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": "Food.com_RAW_rating_ge_4_strict_split",
        "model": name,
        "k_evaluated": K,
        "hyperparams": hyperparams,
        "recall": round(np.mean(results[name]["recalls"]), 4),
        "ndcg": round(np.mean(results[name]["ndcgs"]), 4),
        "coverage": round(coverage, 4)
    })

file_exists = os.path.isfile(runs_file)
with open(runs_file, mode="a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=run_rows[0].keys())
    if not file_exists:
        writer.writeheader()
    writer.writerows(run_rows)

print(f"Logged {len(run_rows)} model runs to {runs_file}!")
print("Notebook complete. Ready for comparison with future variants.")



=== PHASE 9: EXPERIMENT LOGGING ===
Logged 7 model runs to ../runs/runs.csv!
Notebook complete. Ready for comparison with future variants.
